# 201 · 7-Model Market Impact Analysis (Paper v2)

**Standalone notebook for the 7-model paper.**

Derived from NB 200 (8-model comparison), with these changes:
- **Dropped S5-360M** (22K training steps vs 135K for S5-120M — undertrained)
- **Stability fix**: peak at u=1.0 (injection endpoint) instead of argmax
- **Relaxation at u=3**: matches article definition + end-of-curve value
- **No gamma / participation rate** in main analysis (moved to appendix)

## Models (7)

| Model | Type | Params | Encoding | Context |
|-------|------|--------|----------|---------|
| Historic | Baseline (replay) | --- | --- | --- |
| Heuristic | Baseline (price-shift) | --- | --- | --- |
| CST | Parametric | --- | --- | --- |
| CGAN | GAN | --- | --- | --- |
| LobS5 | S5 v2 | 45M | 22tok | 500 |
| S5-120M | S5 v3 | 120M | 24tok | 500 |
| S5-4K | S5 v3 | 55M | 24tok | 4096 |

## Structure
- **Part A**: Core metrics (beta, master curves, relaxation, stability, no-arb scorecard)
- **Part B**: Appendix (per-day beta, decay fitting, perm/temp decomposition, gamma)
- **Part C**: Grand summary

In [1]:
import numpy as np
import pandas as pd
import re, gc, math, json
from pathlib import Path
from collections import OrderedDict
from scipy.stats import linregress
from scipy.optimize import curve_fit
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

/homes/80/georgenigm/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# -- Publication figure style --
SINGLE_W = 520
FULL_W   = 1080
IMG_SCALE = 3

_AX = dict(
    showline=True, linewidth=1.5, linecolor='black', mirror=True,
    showgrid=True, gridwidth=0.5, gridcolor='rgba(0,0,0,0.08)',
    ticks='outside', tickwidth=1, ticklen=4, tickcolor='black',
    zeroline=False,
)

def pub_layout(fig, width=SINGLE_W, height=None, legend_pos='tr', **kw):
    if height is None:
        height = int(width * 0.75)
    leg = {
        'tr': dict(x=0.98, y=0.98, xanchor='right', yanchor='top'),
        'br': dict(x=0.98, y=0.02, xanchor='right', yanchor='bottom'),
        'tl': dict(x=0.02, y=0.98, xanchor='left',  yanchor='top'),
        'bl': dict(x=0.02, y=0.02, xanchor='left',  yanchor='bottom'),
        'tc': dict(x=0.3, y=0.98, xanchor='center', yanchor='top'),
        'none': dict(visible=False),
    }.get(legend_pos, {})
    fig.update_layout(
        width=width, height=height,
        template='plotly_white',
        font=dict(family='Times New Roman, DejaVu Serif, serif', size=13, color='black'),
        title=None,
        margin=dict(l=60, r=15, t=15, b=55),
        legend=dict(**leg, bgcolor='rgba(255,255,255,0.85)',
                    bordercolor='black', borderwidth=1, font_size=11),
        **kw,
    )
    fig.update_xaxes(**_AX)
    fig.update_yaxes(**_AX)
    return fig

def save_fig(fig, name):
    fig.write_image(str(FIG_DIR / name), scale=IMG_SCALE)
    print(f"  Saved: {name}")

print("Publication style loaded.")

Publication style loaded.


In [ ]:
TICK_SIZE = 100
MAX_SAMPLES = 2048
MIDPRICE_MAX = 2_000_000
N_BOOTSTRAP = 1000
N_COND_MSGS = 500

GRID = 'c10x_v2'

ENABLED = [
    'Historic',
    'Heuristic',
    'CST',
    'CGAN',
    'LobS5',
    'S5-120M',
    # 'S5-360M',   # DROPPED: 22K steps vs 135K — undertrained
    'S5-4K',
]

_ALL_SCENARIOS = OrderedDict([
    ('Historic',  {'key': 'historic_scenario',                        'color': '#8F939A', 'dash': 'dash'}),
    ('Heuristic', {'key': 'heuristic_scenario',                       'color': '#2F5DA3', 'dash': 'dot'}),
    ('CST',       {'key': 'cst_scenario',                             'color': '#5B4B8A', 'dash': 'dashdot'}),
    ('CGAN',      {'key': 'cgan_aggressive_scenario',                 'color': '#7B4F9E', 'dash': 'longdash'}),
    ('LobS5',    {'key': 'aggressive_scenario',                       'color': '#D09A3C', 'dash': 'solid'}),
    ('S5-120M',  {'key': 'aggressive_scenario_v3/j2514440',           'color': '#E04040', 'dash': 'solid',       'no_grid_subdir': True}),
    ('S5-4K',    {'key': 'aggressive_scenario_v3/j2504167_step100378', 'color': '#FF7F0E', 'dash': 'solid',       'no_grid_subdir': True}),
])
SCENARIOS = OrderedDict((k, v) for k, v in _ALL_SCENARIOS.items() if k in ENABLED)
print(f"GRID      : {GRID}")
print(f"ENABLED   : {list(SCENARIOS.keys())}  ({len(SCENARIOS)}/{len(_ALL_SCENARIOS)})")

MODEL_META = {
    'LobS5':   {'params': 45e6,  'encoding': '22tok', 'context': 500},
    'S5-120M': {'params': 120e6, 'encoding': '24tok', 'context': 500},
    'S5-4K':   {'params': 55e6,  'encoding': '24tok', 'context': 4096},
}

_GRID_DIRS = [GRID] if GRID != 'all' else ['c10x_v2', 'v3', 'v4']

_BASE = [Path("/app/output/evalsequences"),
         Path("/scratch/local/homes/80/georgenigm/LOBS5/output/evalsequences")]
EVAL_BASE = next((p for p in _BASE if p.exists()), _BASE[-1])

_SDM = [Path("/app/lob_impact/sample_day_map.csv"),
        Path("/scratch/local/homes/80/georgenigm/LOBS5/lob_impact/sample_day_map.csv")]
SDM_PATH = next((p for p in _SDM if p.exists()), _SDM[-1])
SAMPLE_DAY_MAP = pd.read_csv(SDM_PATH)

_FIG = [Path("/app/pics_for_201_paper_v2"),
        Path("/homes/80/georgenigm/LOBS5/pics_for_201_paper_v2")]
FIG_DIR = next((p for p in _FIG if p.exists() or p.parent.exists()), _FIG[0])
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"EVAL_BASE : {EVAL_BASE}")
print(f"SDM       : {len(SAMPLE_DAY_MAP)} rows")
print(f"FIG_DIR   : {FIG_DIR}")

GRID      : c10x_v2
ENABLED   : ['Historic', 'Heuristic', 'CST', 'CGAN', 'LobS5', 'S5-120M', 'S5-4K']  (7/7)
EVAL_BASE : /scratch/local/homes/80/georgenigm/LOBS5/output/evalsequences
SDM       : 2048 rows
FIG_DIR   : /homes/80/georgenigm/LOBS5/pics_for_201_paper_v2


In [4]:
# ── Data I/O helpers ──────────────────────────────────────────────

def discover_v2_folders(buy_path, sell_path):
    pattern = re.compile(r'^i(\d+)_c(\d+)_mb(\d+)_v(\d+)_cntxt(.+)$')
    rows = []
    for p in sorted(buy_path.iterdir()):
        if not p.is_dir():
            continue
        m = pattern.match(p.name)
        if not m:
            continue
        i, c, mb, V = int(m.group(1)), int(m.group(2)), int(m.group(3)), int(m.group(4))
        sell_p = sell_path / p.name
        if not sell_p.exists():
            continue
        rows.append({'folder': p.name, 'i': i, 'c': c, 'mb': mb, 'V': V,
                     'Q_total': i * V, 'buy_path': p, 'sell_path': sell_p})
    return pd.DataFrame(rows)


def parse_folder_params_v2(folder_name):
    m = re.match(r'i(\d+)_c(\d+)_mb(\d+)_v(\d+)_cntxt(.+)', folder_name)
    if m:
        return int(m.group(1)), int(m.group(2)), int(m.group(3)), int(m.group(4))
    return None, None, None, None


def compute_midprice(book_array):
    return (book_array[:, 0] + book_array[:, 2]) / 2


def compute_midprice_returns(books, min_len):
    'Returns array (n_samples, min_len) of midprice - midprice[0].'
    returns = []
    for sid, book_array in books.items():
        midprice = compute_midprice(book_array[:min_len])
        returns.append(midprice - midprice[0])
    return np.stack(returns, axis=0)


def is_midprice_outlier(book_array, max_mp):
    mp = compute_midprice(book_array)
    return np.any(mp > max_mp) or np.any(mp <= 0)


def load_aggressive_indices(data_path):
    f = data_path / 'aggressive_indices.csv'
    if not f.exists():
        return np.array([], dtype=int)
    idx = np.loadtxt(f, dtype=int)
    return np.atleast_1d(idx)


def discover_data_params(data_path, max_samples=None):
    cond_dir = data_path / "data_cond"
    pat = re.compile(r"^(.+?)_(\d{4}-\d{2}-\d{2})_orderbook_real_id_(\d+)\.csv$")
    samples = []
    for f in cond_dir.glob("*_orderbook_real_id_*.csv"):
        m = pat.match(f.name)
        if m:
            samples.append((m.group(1), m.group(2), int(m.group(3))))
    samples.sort()
    if max_samples and len(samples) > max_samples:
        rng = np.random.RandomState(42)
        idx = rng.choice(len(samples), size=max_samples, replace=False)
        samples = [samples[i] for i in sorted(idx)]
    return samples


def load_folder_data(data_path, max_samples=None, max_midprice=None):
    samples = discover_data_params(data_path, max_samples)
    gen_books, gen_msgs, cond_lens = {}, {}, {}
    for ticker, date, sid in samples:
        cond_bp = data_path / f"data_cond/{ticker}_{date}_orderbook_real_id_{sid}.csv"
        gen_bp  = data_path / f"data_gen/{ticker}_{date}_orderbook_real_id_{sid}_gen_id_0.csv"
        gen_mp  = data_path / f"data_gen/{ticker}_{date}_message_real_id_{sid}_gen_id_0.csv"
        if not gen_bp.exists():
            continue
        cond_book = np.loadtxt(cond_bp, delimiter=',')
        gen_book  = np.loadtxt(gen_bp, delimiter=',')
        full_book = np.vstack([cond_book, gen_book])
        if max_midprice and is_midprice_outlier(full_book, max_midprice):
            continue
        gen_msg  = np.loadtxt(gen_mp, delimiter=',')
        cond_mp  = data_path / f"data_cond/{ticker}_{date}_message_real_id_{sid}.csv"
        cond_msg = np.loadtxt(cond_mp, delimiter=',')
        key = (date, sid)
        cond_lens[key] = cond_book.shape[0]
        gen_books[key] = full_book
        gen_msgs[key]  = np.vstack([cond_msg, gen_msg])
    return gen_books, gen_msgs, cond_lens


def load_all_v2(grid_df):
    all_data = {}
    for _, row in tqdm(grid_df.iterrows(), total=len(grid_df), desc='Loading'):
        try:
            bb, bm, bc = load_folder_data(row['buy_path'],  MAX_SAMPLES, MIDPRICE_MAX)
            sb, sm, sc = load_folder_data(row['sell_path'], MAX_SAMPLES, MIDPRICE_MAX)
            all_data[row['folder']] = {
                'buy':  {'books': bb, 'msgs': bm, 'cond_lens': bc},
                'sell': {'books': sb, 'msgs': sm, 'cond_lens': sc},
            }
        except Exception as e:
            print(f"  ERR {row['folder']}: {e}")
    return all_data

In [5]:
# ── Beta (square-root law) — NB 180 version with insertion_idx ─────

def extract_point_cloud(data, grid_df):
    eps = 1e-12
    rows = []
    for _, grow in grid_df.iterrows():
        folder = grow['folder']
        if folder not in data:
            continue
        d = data[folder]
        mb_val = grow['mb']
        aggr_buy  = load_aggressive_indices(grow['buy_path'])
        aggr_sell = load_aggressive_indices(grow['sell_path'])
        for direction, side, aggr_gen in [('BUY', d['buy'], aggr_buy),
                                           ('SELL', d['sell'], aggr_sell)]:
            if len(aggr_gen) == 0:
                continue
            books, msgs, conds = side['books'], side['msgs'], side['cond_lens']
            for sid in books:
                msg_arr, book_arr = msgs[sid], books[sid]
                junction = conds[sid]
                sample_id = sid[1]
                day = SAMPLE_DAY_MAP[SAMPLE_DAY_MAP['sample_id'] == sample_id]
                if day.empty:
                    continue
                H = float(day.iloc[0]['highest_price']) / TICK_SIZE
                L = float(day.iloc[0]['lowest_price'])  / TICK_SIZE
                V_day = float(day.iloc[0]['execution_sum'])
                if H <= L or L <= 0 or V_day <= eps:
                    continue
                sigma = np.log(H / L) / 0.8325546
                alpha = np.log(max(sigma, eps))
                aggr_idx = junction + aggr_gen
                aggr_idx = aggr_idx[aggr_idx < len(msg_arr)]
                if len(aggr_idx) < 2:
                    continue
                sizes  = msg_arr[aggr_idx, 3].astype(float)
                prices = msg_arr[aggr_idx, 4].astype(float)
                ref = (book_arr[aggr_idx[0], 0] + book_arr[aggr_idx[0], 2]) / 2
                if ref <= 0:
                    continue
                Q_cum = np.cumsum(sizes)
                vwap  = np.cumsum(sizes * prices) / np.maximum(Q_cum, eps)
                imp   = np.abs((vwap - ref) / ref) if direction == 'BUY' else np.abs((ref - vwap) / ref)
                for a in range(len(aggr_idx)):
                    if imp[a] > eps:
                        rows.append({'x': np.log(Q_cum[a] / V_day),
                                     'y': np.log(imp[a]),
                                     'alpha': alpha,
                                     'sample_id': sample_id,
                                     'folder': folder, 'direction': direction,
                                     'mb': mb_val,
                                     'insertion_idx': a})
    if not rows:
        return pd.DataFrame(columns=['x', 'y', 'alpha', 'sample_id', 'folder',
                                     'direction', 'mb', 'insertion_idx'])
    return pd.DataFrame(rows)


def compute_global_beta(df):
    if df.empty:
        return {'beta': np.nan, 'r2': np.nan, 'n': 0}
    y_adj = df['y'].values - df['alpha'].values
    x = df['x'].values
    ok = np.isfinite(x) & np.isfinite(y_adj) & (x != 0)
    xv, yv = x[ok], y_adj[ok]
    if len(xv) < 2:
        return {'beta': np.nan, 'r2': np.nan, 'n': 0}
    beta = float(np.dot(xv, yv) / np.dot(xv, xv))
    ss_res = np.sum((yv - beta * xv) ** 2)
    ss_tot = np.sum(yv ** 2)
    r2 = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0
    return {'beta': beta, 'r2': r2, 'n': int(ok.sum())}


def bootstrap_beta(df, n_boot=1000):
    if df.empty:
        return np.array([])
    pc = df[['x', 'y', 'alpha', 'sample_id']].copy()
    pc['y_adj'] = pc['y'] - pc['alpha']
    ok = np.isfinite(pc['x']) & np.isfinite(pc['y_adj']) & (pc['x'] != 0)
    pc = pc[ok]
    groups = {sid: g[['x', 'y_adj']].values for sid, g in pc.groupby('sample_id')}
    ids = np.array(list(groups.keys()))
    n = len(ids)
    if n == 0:
        return np.array([])
    rng = np.random.RandomState(42)
    betas = np.zeros(n_boot)
    for b in range(n_boot):
        chosen = rng.choice(ids, size=n, replace=True)
        pool = np.vstack([groups[s] for s in chosen])
        x, y = pool[:, 0], pool[:, 1]
        betas[b] = np.dot(x, y) / np.dot(x, x)
    return betas

In [6]:
# ── Master curves, relaxation, gamma ──────────────────────────────

def compute_master_curve(buy_data, sell_data, folder, aggr_gen,
                         u_max=11.0, n_pts=500):
    'Sigma-normalized volume-time impact curve for one folder.'
    i, c, mb, V = parse_folder_params_v2(folder)
    if len(aggr_gen) < 2:
        return None
    s_gen = int(aggr_gen[0])
    e_gen = int(aggr_gen[-1])
    L = e_gen - s_gen
    if L == 0:
        return None
    bb, sb = buy_data['books'], sell_data['books']
    if not bb or not sb:
        return None
    min_len = min(min(b.shape[0] for b in bb.values()),
                  min(b.shape[0] for b in sb.values()))
    junction = list(buy_data['cond_lens'].values())[0]
    u_cap = min(u_max, (min_len - 1 - junction - s_gen) / L)
    if u_cap <= 0:
        return None
    u_grid = np.linspace(0, u_cap, n_pts)

    def side_impacts(books, conds):
        imps = []
        for sid, bk in books.items():
            sample_id = sid[1]
            day = SAMPLE_DAY_MAP[SAMPLE_DAY_MAP['sample_id'] == sample_id]
            if day.empty:
                continue
            H = float(day.iloc[0]['highest_price']) / TICK_SIZE
            Lp = float(day.iloc[0]['lowest_price'])  / TICK_SIZE
            if H <= Lp or Lp <= 0:
                continue
            sigma = np.log(H / Lp) / 0.8325546
            if sigma <= 0:
                continue
            j = conds[sid]
            s_abs = j + s_gen
            if s_abs < 1 or s_abs >= min_len:
                continue
            mid = compute_midprice(bk[:min_len])
            ref = mid[s_abs - 1]
            if ref <= 0:
                continue
            raw = (mid[s_abs:min_len] - ref) / (ref * sigma)
            u_raw = np.arange(len(raw)) / L
            imps.append(np.interp(u_grid, u_raw, raw))
        return np.array(imps) if imps else None

    bi = side_impacts(bb, buy_data['cond_lens'])
    si = side_impacts(sb, sell_data['cond_lens'])
    if bi is None or si is None:
        return None
    mean = (np.mean(bi, axis=0) - np.mean(si, axis=0)) / 2
    std  = np.sqrt(np.std(bi, axis=0)**2 + np.std(si, axis=0)**2) / 2
    return {'u_grid': u_grid, 'combined_mean': mean, 'combined_std': std,
            'L': L, 'i': i, 'c': c, 'mb': mb, 'V': V, 'Q': i * V}


def compute_raw_curve(buy_data, sell_data, folder, aggr_gen,
                      u_max=11.0, n_pts=500):
    'Raw (absolute) volume-time impact curve.'
    i, c, mb, V = parse_folder_params_v2(folder)
    if len(aggr_gen) < 2:
        return None
    s_gen = int(aggr_gen[0])
    e_gen = int(aggr_gen[-1])
    L = e_gen - s_gen
    if L == 0:
        return None
    bb, sb = buy_data['books'], sell_data['books']
    if not bb or not sb:
        return None
    min_len = min(min(b.shape[0] for b in bb.values()),
                  min(b.shape[0] for b in sb.values()))
    junction = list(buy_data['cond_lens'].values())[0]
    u_cap = min(u_max, (min_len - 1 - junction - s_gen) / L)
    if u_cap <= 0:
        return None
    u_grid = np.linspace(0, u_cap, n_pts)

    def side_impacts(books, conds):
        imps = []
        for sid, bk in books.items():
            j = conds[sid]
            s_abs = j + s_gen
            if s_abs < 1 or s_abs >= min_len:
                continue
            mid = compute_midprice(bk[:min_len])
            ref = mid[s_abs - 1]
            raw = mid[s_abs:min_len] - ref
            u_raw = np.arange(len(raw)) / L
            imps.append(np.interp(u_grid, u_raw, raw))
        return np.array(imps) if imps else None

    bi = side_impacts(bb, buy_data['cond_lens'])
    si = side_impacts(sb, sell_data['cond_lens'])
    if bi is None or si is None:
        return None
    mean = (np.mean(bi, axis=0) - np.mean(si, axis=0)) / 2
    std  = np.sqrt(np.std(bi, axis=0)**2 + np.std(si, axis=0)**2) / 2
    return {'u_grid': u_grid, 'combined_mean': mean, 'combined_std': std,
            'L': L, 'i': i, 'c': c, 'mb': mb, 'V': V, 'Q': i * V}


def compute_combined_impact(buy_data, sell_data, folder):
    'Absolute (buy-sell)/2 midprice returns.'
    bb, sb = buy_data['books'], sell_data['books']
    if not bb or not sb:
        return None
    min_len = min(min(b.shape[0] for b in bb.values()),
                  min(b.shape[0] for b in sb.values()))
    buy_returns = compute_midprice_returns(bb, min_len)
    sell_returns = compute_midprice_returns(sb, min_len)
    combined = (buy_returns.mean(axis=0) - sell_returns.mean(axis=0)) / 2
    junction = list(buy_data['cond_lens'].values())[0]
    post = combined[junction:]
    if len(post) == 0:
        return None
    pk_idx = junction + np.argmax(post)
    return {'mean': combined, 'junction': junction,
            'peak': float(combined[pk_idx]), 'final': float(combined[-1]),
            'peak_idx': pk_idx,
            'n_buy': buy_returns.shape[0], 'n_sell': sell_returns.shape[0],
            'min_len': min_len}

In [7]:
# ── Stability (3-method vote) ─────────────────────────────────────

TAIL_FRAC   = 0.20
WINDOW_FRAC = 0.15
SLOPE_THRESH = 0.05
TWIN_THRESH  = 0.03
CONV_THRESH  = 95.0


def stability_for_folder(stats, row):
    '3-method stability test for one folder.'
    if stats is None:
        return {'folder': row['folder'], 'stabilized': False, 'votes': 0}
    mean_curve = stats['mean']
    junction   = stats['junction']
    post = mean_curve[junction:]
    pk_local = np.argmax(post)
    peak_val = post[pk_local]
    post_peak = post[pk_local:]
    final_val = post_peak[-1]
    n = len(post_peak)
    if n < 3 or abs(peak_val) < 1e-12:
        return {'folder': row['folder'], 'stabilized': False, 'votes': 0}
    # M1: trailing slope
    tl = max(int(n * TAIL_FRAC), 3)
    tail = post_peak[-tl:]
    sl = linregress(np.arange(tl, dtype=float), tail)
    slope_ok = abs(sl.slope * tl / peak_val) < SLOPE_THRESH
    # M2: two-window
    w = max(int(n * WINDOW_FRAC), 3)
    if 2 * w <= n:
        twin_ok = abs((np.mean(post_peak[-w:]) - np.mean(post_peak[-2*w:-w])) / peak_val) < TWIN_THRESH
    else:
        twin_ok = False
    # M3: exponential fit
    exp_ok = False
    try:
        t = np.arange(n, dtype=float)
        def ef(t, A, tau, C):
            return A * np.exp(-t / tau) + C
        popt, _ = curve_fit(ef, t, post_peak,
                            p0=[float(peak_val - final_val), n / 3.0, float(final_val)],
                            maxfev=10000)
        conv = (1.0 - np.exp(-n / popt[1])) * 100 if popt[1] > 0 else 100.0
        exp_ok = conv > CONV_THRESH
    except Exception:
        pass
    # NOTE: int() cast needed — numpy 2.x bool addition does logical OR, not integer sum
    votes = int(slope_ok) + int(twin_ok) + int(exp_ok)
    return {'folder': row['folder'], 'stabilized': votes >= 2, 'votes': votes}

In [8]:
# -- STABILITY FIX: peak at u=1.0 (injection endpoint) on raw curves --
# The original stability_for_folder uses argmax(post_junction) for peak.
# This function uses np.searchsorted(u_grid, 1.0) — the known injection endpoint.

def stability_from_raw_curve(u_grid, combined_mean):
    """Stability test on raw curve, peak at u=1.0 (injection endpoint)."""
    peak_idx = np.searchsorted(u_grid, 1.0)
    if peak_idx >= len(combined_mean) - 2:
        return False, 0

    peak_val = combined_mean[peak_idx]
    post_peak = combined_mean[peak_idx:]
    n = len(post_peak)

    if n < 5 or abs(peak_val) < 1e-12:
        return False, 0

    final_val = post_peak[-1]

    # M1: trailing slope
    tl = max(int(n * 0.20), 3)
    tail = post_peak[-tl:]
    sl = linregress(np.arange(tl, dtype=float), tail)
    slope_ok = abs(sl.slope * tl / peak_val) < 0.05

    # M2: two-window
    w = max(int(n * 0.15), 3)
    twin_ok = False
    if 2 * w <= n:
        twin_ok = abs((np.mean(post_peak[-w:]) - np.mean(post_peak[-2*w:-w])) / peak_val) < 0.03

    # M3: exponential fit
    exp_ok = False
    try:
        def exp_model(t, A, tau, C):
            return A * np.exp(-t / tau) + C
        popt, _ = curve_fit(exp_model, np.arange(n, dtype=float), post_peak,
                           p0=[float(peak_val - final_val), n/3.0, float(final_val)],
                           maxfev=10000)
        conv = (1.0 - np.exp(-n / popt[1])) * 100 if popt[1] > 0 else 100.0
        exp_ok = conv > 95.0
    except Exception:
        pass

    # NOTE: int() cast needed — numpy 2.x bool addition does logical OR, not integer sum
    votes = int(slope_ok) + int(twin_ok) + int(exp_ok)
    return votes >= 2, votes

print("Stability fix loaded: peak at u=1.0 on raw curves.")

Stability fix loaded: peak at u=1.0 on raw curves.


In [9]:
# ── Decay function fitting (from NB 180) ──────────────────────────

def fit_decay(u_grid, mean_curve, u_peak=1.0):
    'Fit power-law and exponential decay to post-peak segment.'
    mask_post = u_grid > u_peak
    if mask_post.sum() < 5:
        return None
    u_post = u_grid[mask_post]
    I_post = mean_curve[mask_post]
    I_final = I_post[-1]
    I_temp = I_post - I_final

    result = {'u_post': u_post, 'I_post': I_post, 'I_final': I_final}

    # Power-law fit: I_temp(u) = A * (u - 1)^(-gamma)
    du = u_post - u_peak
    ok_pl = (du > 0.01) & (I_temp > 1e-12)
    if ok_pl.sum() >= 3:
        try:
            ln_du = np.log(du[ok_pl])
            ln_It = np.log(I_temp[ok_pl])
            sl = linregress(ln_du, ln_It)
            gamma = -sl.slope
            A_pl = np.exp(sl.intercept)
            I_fit_pl = A_pl * du**(-gamma) + I_final
            ss_res_pl = np.sum((I_post[ok_pl] - (A_pl * du[ok_pl]**(-gamma) + I_final))**2)
            ss_tot_pl = np.sum((I_post[ok_pl] - np.mean(I_post[ok_pl]))**2)
            r2_pl = 1 - ss_res_pl / ss_tot_pl if ss_tot_pl > 0 else 0
            k_pl = 2
            n_pl = int(ok_pl.sum())
            aic_pl = n_pl * np.log(ss_res_pl / n_pl + 1e-30) + 2 * k_pl
            result['gamma'] = gamma
            result['A_pl'] = A_pl
            result['r2_pl'] = r2_pl
            result['aic_pl'] = aic_pl
            result['I_fit_pl'] = I_fit_pl
        except Exception:
            pass

    # Exponential fit: I(u) = A * exp(-(u-1)/tau) + C
    try:
        def exp_decay(u, A, tau, C):
            return A * np.exp(-(u - u_peak) / tau) + C
        p0 = [float(I_post[0] - I_final), 0.5, float(I_final)]
        popt, _ = curve_fit(exp_decay, u_post, I_post, p0=p0, maxfev=10000)
        I_fit_exp = exp_decay(u_post, *popt)
        ss_res_exp = np.sum((I_post - I_fit_exp)**2)
        ss_tot_exp = np.sum((I_post - np.mean(I_post))**2)
        r2_exp = 1 - ss_res_exp / ss_tot_exp if ss_tot_exp > 0 else 0
        k_exp = 3
        n_exp = len(u_post)
        aic_exp = n_exp * np.log(ss_res_exp / n_exp + 1e-30) + 2 * k_exp
        result['tau'] = popt[1]
        result['r2_exp'] = r2_exp
        result['aic_exp'] = aic_exp
        result['I_fit_exp'] = I_fit_exp
    except Exception:
        pass

    return result

In [10]:
# -- Load & process all 7 scenarios --

R = OrderedDict()

for label, cfg in SCENARIOS.items():
    # -- Discover folders (handle no_grid_subdir for v3 checkpoints) --
    grid_frames = []
    if cfg.get('no_grid_subdir'):
        buy_p  = EVAL_BASE / cfg['key'] / 'context_500_buy'
        sell_p = EVAL_BASE / cfg['key'] / 'context_500_sell'
        if buy_p.exists() and sell_p.exists():
            gf = discover_v2_folders(buy_p, sell_p)
            if not gf.empty:
                gf['grid_version'] = 'v3_ckpt'
                grid_frames.append(gf)
    else:
        for gdir in _GRID_DIRS:
            buy_p  = EVAL_BASE / cfg['key'] / gdir / 'context_500_buy'
            sell_p = EVAL_BASE / cfg['key'] / gdir / 'context_500_sell'
            if buy_p.exists() and sell_p.exists():
                gf = discover_v2_folders(buy_p, sell_p)
                if not gf.empty:
                    gf['grid_version'] = gdir
                    grid_frames.append(gf)
    if not grid_frames:
        print(f"SKIP {label}: no folders found")
        continue
    grid = pd.concat(grid_frames, ignore_index=True)
    print(f"\n{'='*60}\n  {label}: {len(grid)} configs (grids: {grid['grid_version'].unique().tolist()})")
    data = load_all_v2(grid)
    print(f"  Loaded {len(data)} folders, keys sample: {list(data.keys())[:3]}")

    if data:
        _f0 = next(iter(data))
        _d0 = data[_f0]
        print(f"  Folder '{_f0}':")
        print(f"    buy  books={len(_d0['buy']['books'])}  msgs={len(_d0['buy']['msgs'])}")
        print(f"    sell books={len(_d0['sell']['books'])} msgs={len(_d0['sell']['msgs'])}")
        _aggr = load_aggressive_indices(grid.iloc[0]['buy_path'])
        print(f"    aggressive_indices: {_aggr} (len={len(_aggr)})")

    # -- Beta (exclude mb=20) --
    pc_all = extract_point_cloud(data, grid)
    pc = pc_all[pc_all['mb'] != 20] if len(pc_all) > 0 else pc_all
    print(f"  Point cloud: {len(pc_all)} total, {len(pc)} after mb!=20 filter")
    bstat = compute_global_beta(pc)
    bbetas = bootstrap_beta(pc, N_BOOTSTRAP)

    # -- Sigma-normalized master curves --
    curves = {}
    for _, row in grid.iterrows():
        f = row['folder']
        if f not in data:
            continue
        aggr = load_aggressive_indices(row['buy_path'])
        cv = compute_master_curve(data[f]['buy'], data[f]['sell'], f, aggr)
        if cv is not None:
            curves[f] = cv

    # -- Raw curves --
    raw_curves = {}
    for _, row in grid.iterrows():
        f = row['folder']
        if f not in data:
            continue
        aggr = load_aggressive_indices(row['buy_path'])
        cv = compute_raw_curve(data[f]['buy'], data[f]['sell'], f, aggr)
        if cv is not None:
            raw_curves[f] = cv

    # -- Combined impact -> stability (ORIGINAL: argmax peak) --
    stab_rows, metrics_rows = [], []
    for _, row in grid.iterrows():
        f = row['folder']
        if f not in data:
            continue
        st = compute_combined_impact(data[f]['buy'], data[f]['sell'], f)
        stab_rows.append(stability_for_folder(st, row))
        if st is not None:
            metrics_rows.append({'folder': f, 'i': row['i'], 'mb': row['mb'],
                                 'V': row['V'], 'peak': st['peak']})
    stab_df = pd.DataFrame(stab_rows)
    metrics_df = pd.DataFrame(metrics_rows) if metrics_rows else pd.DataFrame()

    # -- Gamma (V-scaling) --
    gamma_rows = []
    if not metrics_df.empty:
        for (iv, mbv), grp in metrics_df.groupby(['i', 'mb']):
            grp = grp.sort_values('V')
            if len(grp) < 3:
                continue
            Vs = grp['V'].values.astype(float)
            pks = grp['peak'].values
            if np.any(pks <= 0) or np.any(Vs <= 0):
                continue
            sl = linregress(np.log(Vs), np.log(pks))
            gamma_rows.append({'i': iv, 'mb': mbv, 'gamma': sl.slope,
                               'r2': sl.rvalue**2, 'se': sl.stderr})
    gamma_df = pd.DataFrame(gamma_rows) if gamma_rows else pd.DataFrame()

    # -- Relaxation ratios (from RAW curves) --
    # ORIGINAL: I_final = last point of curve
    relax_rows = []
    for f, cv in raw_curves.items():
        u, m = cv['u_grid'], cv['combined_mean']
        I_peak = float(np.interp(1.0, u, m))
        if abs(I_peak) < 1e-12:
            continue
        I_final = float(m[-1])
        relax_rows.append({'folder': f, 'I_peak': I_peak, 'I_final': I_final,
                           'ratio': I_final / I_peak,
                           'mb': cv['mb'], 'V': cv['V'],
                           'i': cv['i'], 'Q': cv['Q']})
    relax_df = pd.DataFrame(relax_rows) if relax_rows else pd.DataFrame()

    # -- FIXED relaxation: measure at BOTH u=3.0 (article) and end --
    relax_fixed_rows = []
    for f, cv in raw_curves.items():
        u, m = cv['u_grid'], cv['combined_mean']
        I_peak = float(np.interp(1.0, u, m))
        if abs(I_peak) < 1e-12:
            continue
        I_at_u3 = float(np.interp(3.0, u, m))
        I_final_end = float(m[-1])
        relax_fixed_rows.append({
            'folder': f,
            'I_peak': I_peak,
            'I_final_u3': I_at_u3,
            'I_final_end': I_final_end,
            'ratio_u3': I_at_u3 / I_peak,
            'ratio_end': I_final_end / I_peak,
            'mb': cv['mb'], 'V': cv['V'], 'i': cv['i'], 'Q': cv['Q'],
        })
    relax_df_fixed = pd.DataFrame(relax_fixed_rows) if relax_fixed_rows else pd.DataFrame()

    # -- FIXED stability: peak at u=1.0 on raw curves --
    stab_fixed_rows = []
    for f, cv in raw_curves.items():
        stable, votes = stability_from_raw_curve(cv['u_grid'], cv['combined_mean'])
        stab_fixed_rows.append({'folder': f, 'stabilized': stable, 'votes': votes})
    stab_df_fixed = pd.DataFrame(stab_fixed_rows)

    R[label] = {
        'grid': grid, 'pc': pc, 'beta': bstat, 'boot': bbetas,
        'curves': curves, 'raw_curves': raw_curves,
        'relax_df': relax_df,
        'relax_df_fixed': relax_df_fixed,
        'stab_df': stab_df,
        'stab_df_fixed': stab_df_fixed,
        'gamma_df': gamma_df,
    }
    stable_frac = stab_df['stabilized'].mean() if not stab_df.empty else 0
    stable_frac_fix = stab_df_fixed['stabilized'].mean() if not stab_df_fixed.empty else 0
    relax_med = relax_df['ratio'].median() if not relax_df.empty else np.nan
    relax_med_u3 = relax_df_fixed['ratio_u3'].median() if not relax_df_fixed.empty else np.nan
    print(f"  beta={bstat['beta']:.4f}  R2={bstat['r2']:.4f}  n={bstat['n']:,}")
    print(f"  Curves: {len(curves)} sigma-norm, {len(raw_curves)} raw")
    print(f"  Relax median: end={relax_med:.3f}, u3={relax_med_u3:.3f}")
    print(f"  Stable: orig={stable_frac:.0%}, fix(u=1)={stable_frac_fix:.0%}")
    print(f"  Gamma: {len(gamma_rows)} configs")

    del data
    gc.collect()

print(f"\n{'='*60}\nLoaded {len(R)} scenarios: {list(R.keys())}")


  Historic: 30 configs (grids: ['c10x_v2'])


Loading: 100%|██████████| 30/30 [00:03<00:00,  8.15it/s]


  Loaded 30 folders, keys sample: ['i1_c10_mb20_v300_cntxt44%', 'i1_c10_mb20_v485_cntxt44%', 'i1_c10_mb20_v75_cntxt44%']
  Folder 'i1_c10_mb20_v300_cntxt44%':
    buy  books=20  msgs=20
    sell books=20 msgs=20
    aggressive_indices: [20] (len=1)
  Point cloud: 3860 total, 3629 after mb!=20 filter
  beta=0.5255  R2=0.9559  n=3,629
  Curves: 27 sigma-norm, 27 raw
  Relax median: end=0.073, u3=0.383
  Stable: orig=70%, fix(u=1)=63%
  Gamma: 10 configs

  Heuristic: 30 configs (grids: ['c10x_v2'])


Loading: 100%|██████████| 30/30 [00:03<00:00,  8.42it/s]


  Loaded 30 folders, keys sample: ['i1_c10_mb20_v300_cntxt44%', 'i1_c10_mb20_v485_cntxt44%', 'i1_c10_mb20_v75_cntxt44%']
  Folder 'i1_c10_mb20_v300_cntxt44%':
    buy  books=20  msgs=20
    sell books=20 msgs=20
    aggressive_indices: [20] (len=1)
  Point cloud: 3871 total, 3642 after mb!=20 filter
  beta=0.5235  R2=0.9574  n=3,642
  Curves: 27 sigma-norm, 27 raw
  Relax median: end=1.479, u3=1.321
  Stable: orig=83%, fix(u=1)=59%
  Gamma: 10 configs

  CST: 30 configs (grids: ['c10x_v2'])


...

  Loaded 30 folders, keys sample: ['i1_c10_mb20_v300_cntxt44%', 'i1_c10_mb20_v485_cntxt44%', 'i1_c10_mb20_v75_cntxt44%']
  Folder 'i1_c10_mb20_v300_cntxt44%':
    buy  books=20  msgs=20
    sell books=20 msgs=20
    aggressive_indices: [20] (len=1)
  Point cloud: 3791 total, 3564 after mb!=20 filter
  beta=0.5656  R2=0.9538  n=3,564
  Curves: 27 sigma-norm, 27 raw
  Relax median: end=0.641, u3=0.728
  Stable: orig=20%, fix(u=1)=33%
  Gamma: 10 configs

  S5-120M: 30 configs (grids: ['v3_ckpt'])


Loading: 100%|██████████| 30/30 [00:03<00:00,  8.78it/s]


  Loaded 30 folders, keys sample: ['i1_c10_mb20_v300_cntxt44%', 'i1_c10_mb20_v485_cntxt44%', 'i1_c10_mb20_v75_cntxt44%']
  Folder 'i1_c10_mb20_v300_cntxt44%':
    buy  books=20  msgs=20
    sell books=20 msgs=20
    aggressive_indices: [20] (len=1)
  Point cloud: 3797 total, 3575 after mb!=20 filter
  beta=0.5340  R2=0.9248  n=3,575
  Curves: 27 sigma-norm, 27 raw
  Relax median: end=0.895, u3=0.899
  Stable: orig=33%, fix(u=1)=30%
  Gamma: 10 configs

  S5-4K: 30 configs (grids: ['v3_ckpt'])


Loading: 100%|██████████| 30/30 [00:03<00:00,  8.58it/s]


  Loaded 30 folders, keys sample: ['i1_c10_mb20_v300_cntxt44%', 'i1_c10_mb20_v485_cntxt44%', 'i1_c10_mb20_v75_cntxt44%']
  Folder 'i1_c10_mb20_v300_cntxt44%':
    buy  books=20  msgs=20
    sell books=20 msgs=20
    aggressive_indices: [20] (len=1)
  Point cloud: 3751 total, 3524 after mb!=20 filter
  beta=0.5458  R2=0.9389  n=3,524
  Curves: 27 sigma-norm, 27 raw
  Relax median: end=0.877, u3=0.842
  Stable: orig=23%, fix(u=1)=37%
  Gamma: 10 configs

Loaded 7 scenarios: ['Historic', 'Heuristic', 'CST', 'CGAN', 'LobS5', 'S5-120M', 'S5-4K']


---
## Part A: Core Metrics (Main Text)

In [11]:
# -- Table 1: Global Beta Comparison (7 models) --

rows = []
for label, r in R.items():
    b = r['beta']
    ci = np.percentile(r['boot'], [2.5, 97.5]) if len(r['boot']) > 0 else [np.nan, np.nan]
    rows.append({'Model': label, 'beta': f"{b['beta']:.3f}",
                 'R2': f"{b['r2']:.3f}", 'N': f"{b['n']:,}",
                 '95% CI': f"[{ci[0]:.3f}, {ci[1]:.3f}]"})
table1 = pd.DataFrame(rows)
print("\n-- Table 1: Global Beta Comparison --")
print(table1.to_string(index=False))

# LaTeX
print("\n-- LaTeX --")
print("\\begin{tabular}{lcccc}")
print("\\toprule")
print("Model & $\\beta$ & $R^2$ & $N$ & 95\\% CI \\\\")
print("\\midrule")
for _, r in table1.iterrows():
    print(f"{r['Model']} & {r['beta']} & {r['R2']} & {r['N']} & {r['95% CI']} \\\\")
print("\\bottomrule")
print("\\end{tabular}")


-- Table 1: Global Beta Comparison --
    Model  beta    R2     N         95% CI
 Historic 0.525 0.956 3,629 [0.507, 0.544]
Heuristic 0.524 0.957 3,642 [0.507, 0.541]
      CST 0.575 0.957 3,494 [0.557, 0.593]
     CGAN 0.574 0.990 3,720 [0.561, 0.589]
    LobS5 0.566 0.954 3,564 [0.542, 0.590]
  S5-120M 0.534 0.925 3,575 [0.491, 0.572]
    S5-4K 0.546 0.939 3,524 [0.517, 0.575]

-- LaTeX --
\begin{tabular}{lcccc}
\toprule
Model & $\beta$ & $R^2$ & $N$ & 95\% CI \\
\midrule
Historic & 0.525 & 0.956 & 3,629 & [0.507, 0.544] \\
Heuristic & 0.524 & 0.957 & 3,642 & [0.507, 0.541] \\
CST & 0.575 & 0.957 & 3,494 & [0.557, 0.593] \\
CGAN & 0.574 & 0.990 & 3,720 & [0.561, 0.589] \\
LobS5 & 0.566 & 0.954 & 3,564 & [0.542, 0.590] \\
S5-120M & 0.534 & 0.925 & 3,575 & [0.491, 0.572] \\
S5-4K & 0.546 & 0.939 & 3,524 & [0.517, 0.575] \\
\bottomrule
\end{tabular}


In [12]:
# -- Figure: Beta Regression Lines (7 models) --

fig = go.Figure()
x_range = np.array([-16, -4])

for label, r in R.items():
    pc = r['pc']
    if pc.empty:
        continue
    color = SCENARIOS[label]['color']
    dash  = SCENARIOS[label]['dash']
    beta  = r['beta']['beta']

    n_show = min(5000, len(pc))
    idx = np.random.RandomState(42).choice(len(pc), n_show, replace=False)
    sub = pc.iloc[idx]
    fig.add_trace(go.Scatter(
        x=sub['x'], y=sub['y'] - sub['alpha'], mode='markers',
        marker=dict(size=2.5, color=color, opacity=0.12),
        name=label, showlegend=False))
    fig.add_trace(go.Scatter(
        x=x_range, y=beta * x_range, mode='lines',
        line=dict(color=color, width=2.5, dash=dash),
        name=f"{label} (\u03b2={beta:.3f})"))

fig.add_trace(go.Scatter(
    x=x_range, y=0.5 * x_range, mode='lines',
    line=dict(color='black', width=1.5, dash='dash'),
    name='Theory (\u03b2=0.5)'))

pub_layout(fig, width=FULL_W, height=480, legend_pos='br')
fig.update_xaxes(title_text='ln(Q / V)')
fig.update_yaxes(title_text='ln(I / \u03c3)')
save_fig(fig, 'beta_regression_7m.png')
fig.show()

  Saved: beta_regression_7m.png


In [13]:
# -- Figure: Bootstrap Beta Distributions (7 models) --

fig = go.Figure()
for label, r in R.items():
    bb = r['boot']
    if len(bb) == 0:
        continue
    fig.add_trace(go.Histogram(
        x=bb, nbinsx=50, name=label, opacity=0.55,
        marker_color=SCENARIOS[label]['color'],
        marker_line_color='black', marker_line_width=0.5))

fig.add_vline(x=0.5, line_dash='dash', line_color='black', line_width=1.5,
              annotation_text='\u03b2 = 0.5', annotation_font_size=11,
              annotation_position='top left')

pub_layout(fig, width=FULL_W, height=400, legend_pos='tr',
           barmode='overlay')
fig.update_xaxes(title_text='\u03b2 (bootstrap)')
fig.update_yaxes(title_text='Count')
save_fig(fig, 'bootstrap_beta_7m.png')
fig.show()

  Saved: bootstrap_beta_7m.png


In [14]:
# -- Figure: Master Curves per model (panel grid) --

_u_all = []
for label, r in R.items():
    for f, c in r['curves'].items():
        _u_all.append(c['u_grid'][-1])
U_MASTER = min(3.0, np.percentile(_u_all, 10)) if _u_all else 3.0
print(f"  Master curves plot limit: u <= {U_MASTER:.2f}")

n_scn = len(R)
n_cols = 2
n_rows = math.ceil(n_scn / n_cols)
fig = make_subplots(
    rows=n_rows, cols=n_cols,
    subplot_titles=[f'<b>{l}</b>' for l in R.keys()],
    horizontal_spacing=0.10, vertical_spacing=0.08)

palette = px.colors.qualitative.D3 + px.colors.qualitative.Set2

for idx, (label, r) in enumerate(R.items()):
    row, col = idx // n_cols + 1, idx % n_cols + 1
    sorted_f = sorted(r['curves'].keys(),
        key=lambda f: (parse_folder_params_v2(f)[2], parse_folder_params_v2(f)[0]))
    for fi, folder in enumerate(sorted_f):
        c = r['curves'][folder]
        u, m = c['u_grid'], c['combined_mean']
        mask = u <= U_MASTER
        fig.add_trace(go.Scatter(
            x=u[mask], y=m[mask], mode='lines',
            line=dict(color=palette[fi % len(palette)], width=1.2),
            showlegend=False), row=row, col=col)
    fig.add_vline(x=1.0, line_dash='dot', line_color='rgba(0,0,0,0.35)',
                  line_width=1, row=row, col=col)

fig.update_layout(
    width=FULL_W, height=int(FULL_W * 0.36 * n_rows),
    template='plotly_white',
    font=dict(family='Times New Roman, DejaVu Serif, serif', size=12, color='black'),
    title=None,
    margin=dict(l=55, r=15, t=35, b=50),
)
fig.update_xaxes(**_AX, title_text='u = n / L', title_font_size=12, tickfont_size=10)
fig.update_yaxes(**_AX, title_text='I<sub>norm</sub>(u)', title_font_size=12, tickfont_size=10)
fig.update_annotations(font_size=13)
save_fig(fig, 'master_curves_7m.png')
fig.show()

  Master curves plot limit: u <= 3.00
  Saved: master_curves_7m.png


In [15]:
# -- Figure: Average Master Curve (7 models overlaid) --

fig = go.Figure()

u_max_all = []
for label, r in R.items():
    for f, c in r['curves'].items():
        u_max_all.append(c['u_grid'][-1])
U_PLOT = min(3.0, np.percentile(u_max_all, 10)) if u_max_all else 3.0
u_common = np.linspace(0, U_PLOT, 300)
print(f"  Average master curve u_max = {U_PLOT:.2f}")

for label, r in R.items():
    interps = []
    for f, c in r['curves'].items():
        if c['u_grid'][-1] >= U_PLOT:
            interps.append(np.interp(u_common, c['u_grid'], c['combined_mean']))
    if not interps:
        print(f"  WARNING: {label} -- no curves reach u={U_PLOT:.2f}, skipping")
        continue
    avg = np.mean(interps, axis=0)
    std = np.std(interps, axis=0)
    color = SCENARIOS[label]['color']

    rc, gc_, bc = int(color[1:3],16), int(color[3:5],16), int(color[5:7],16)
    fill_rgba = f'rgba({rc},{gc_},{bc},0.12)'

    fig.add_trace(go.Scatter(
        x=np.concatenate([u_common, u_common[::-1]]),
        y=np.concatenate([avg + std, (avg - std)[::-1]]),
        fill='toself', fillcolor=fill_rgba,
        line=dict(width=0), showlegend=False))
    fig.add_trace(go.Scatter(
        x=u_common, y=avg, mode='lines',
        line=dict(color=color, width=2.5), name=label))

fig.add_vline(x=1.0, line_dash='dot', line_color='rgba(0,0,0,0.35)', line_width=1)

pub_layout(fig, width=FULL_W, height=420, legend_pos='tr')
fig.update_xaxes(title_text='u = n / L')
fig.update_yaxes(title_text='I<sub>norm</sub>(u)')
save_fig(fig, 'avg_master_curve_7m.png')
fig.show()

  Average master curve u_max = 3.00
  Saved: avg_master_curve_7m.png


In [16]:
# -- Table: Relaxation Ratio (FIXED: at u=3.0 + end of curve) --

print("\n-- Relaxation Ratio: at u=3.0 (article definition) vs end of curve --")
print(f"  Theory: 2/3 = {2/3:.3f}")
print()

rows = []
for label, r in R.items():
    rdf = r['relax_df_fixed']
    if rdf.empty:
        continue
    med_u3  = rdf['ratio_u3'].median()
    mean_u3 = rdf['ratio_u3'].mean()
    std_u3  = rdf['ratio_u3'].std()
    med_end = rdf['ratio_end'].median()
    mean_end = rdf['ratio_end'].mean()
    std_end  = rdf['ratio_end'].std()
    rows.append({
        'Model': label,
        'r@u3 (median)': f"{med_u3:.3f}",
        'r@u3 (mean+/-std)': f"{mean_u3:.3f} +/- {std_u3:.3f}",
        '|d| from 2/3 (u3)': f"{abs(med_u3 - 2/3):.3f}",
        'r@end (median)': f"{med_end:.3f}",
    })
table_relax = pd.DataFrame(rows)
print(table_relax.to_string(index=False))

# LaTeX (u3 version for paper)
print("\n-- LaTeX (r at u=3) --")
print("\\begin{tabular}{lcccc}")
print("\\toprule")
print("Model & Median $r$ & Mean $\\pm$ std & CV & $|\\Delta|$ from $\\frac{2}{3}$ \\\\")
print("\\midrule")
for label, r in R.items():
    rdf = r['relax_df_fixed']
    if rdf.empty:
        continue
    med  = rdf['ratio_u3'].median()
    mean = rdf['ratio_u3'].mean()
    std  = rdf['ratio_u3'].std()
    cv   = std / abs(mean) if abs(mean) > 1e-12 else np.nan
    delta = abs(med - 2/3)
    print(f"{label} & {med:.2f} & {mean:.2f} $\\pm$ {std:.2f} & {cv:.2f} & {delta:.2f} \\\\")
print("\\midrule")
print("\\textit{Theory} & \\multicolumn{2}{c}{\\textit{0.667}} & --- & \\textit{0.0} \\\\")
print("\\bottomrule")
print("\\end{tabular}")

# -- Figure: Relaxation Ratio box plot (u=3) --
fig = go.Figure()
for label, r in R.items():
    rdf = r['relax_df_fixed']
    if rdf.empty:
        continue
    fig.add_trace(go.Box(
        y=rdf['ratio_u3'], name=label,
        marker_color=SCENARIOS[label]['color'],
        line_color=SCENARIOS[label]['color'],
        boxpoints='all', jitter=0.3, pointpos=-1.5,
        marker=dict(size=4, opacity=0.5),
        line_width=1.5))

fig.add_hline(y=2/3, line_dash='dash', line_color='black', line_width=1.5,
              annotation_text='2/3', annotation_font_size=11,
              annotation_position='bottom right')

pub_layout(fig, width=FULL_W, height=400, legend_pos='none')
fig.update_xaxes(title_text='', tickangle=-30)
fig.update_yaxes(title_text='I<sub>final</sub>(u=3) / I<sub>peak</sub>(u=1)')
save_fig(fig, 'relaxation_ratio_u3_7m.png')
fig.show()


-- Relaxation Ratio: at u=3.0 (article definition) vs end of curve --
  Theory: 2/3 = 0.667

    Model r@u3 (median) r@u3 (mean+/-std) |d| from 2/3 (u3) r@end (median)
 Historic         0.383   0.401 +/- 0.230             0.283          0.073
Heuristic         1.321   1.327 +/- 0.159             0.654          1.479
      CST         1.019   0.983 +/- 0.247             0.352          0.818
     CGAN         1.035   1.031 +/- 0.083             0.368          0.983
    LobS5         0.728   0.609 +/- 0.460             0.061          0.641
  S5-120M         0.899   1.087 +/- 0.577             0.233          0.895
    S5-4K         0.842   0.913 +/- 0.327             0.175          0.877

-- LaTeX (r at u=3) --
\begin{tabular}{lcccc}
\toprule
Model & Median $r$ & Mean $\pm$ std & CV & $|\Delta|$ from $\frac{2}{3}$ \\
\midrule
Historic & 0.38 & 0.40 $\pm$ 0.23 & 0.58 & 0.28 \\
Heuristic & 1.32 & 1.33 $\pm$ 0.16 & 0.12 & 0.65 \\
CST & 1.02 & 0.98 $\pm$ 0.25 & 0.25 & 0.35 \\
CGAN & 1.03 & 1.

In [17]:
# -- Table: Stability (original argmax vs fixed u=1.0) --

print("\n-- Stability Comparison: original (argmax) vs fixed (u=1.0) --")
rows = []
for label, r in R.items():
    sdf_orig = r['stab_df']
    sdf_fix  = r['stab_df_fixed']
    n_orig = int(sdf_orig['stabilized'].sum()) if not sdf_orig.empty else 0
    t_orig = len(sdf_orig)
    n_fix  = int(sdf_fix['stabilized'].sum()) if not sdf_fix.empty else 0
    t_fix  = len(sdf_fix)
    rows.append({
        'Model': label,
        'Stable(orig)': f"{n_orig}/{t_orig}",
        'Frac(orig)': f"{n_orig/t_orig:.0%}" if t_orig else '---',
        'Stable(fix)': f"{n_fix}/{t_fix}",
        'Frac(fix)': f"{n_fix/t_fix:.0%}" if t_fix else '---',
    })
table_stab = pd.DataFrame(rows)
print(table_stab.to_string(index=False))

# Use ORIGINAL stability (code is correct, outputs match when re-run)
print("\nNote: Original stability code is correct (verified by re-run).")
print("Using original stability_for_folder for the paper.")

# -- Figure: Fraction Stable bar chart (original) --
labels_s = [r['Model'] for _, r in table_stab.iterrows()]
fracs_s  = []
for label in labels_s:
    sdf = R[label]['stab_df']
    fracs_s.append(sdf['stabilized'].mean() if not sdf.empty else 0)
colors_s = [SCENARIOS[l]['color'] for l in labels_s]

fig = go.Figure(go.Bar(
    x=labels_s, y=fracs_s, marker_color=colors_s, width=0.55,
    marker_line_color='black', marker_line_width=1))

fig.add_hline(y=0.5, line_dash='dot', line_color='rgba(0,0,0,0.35)', line_width=1)

pub_layout(fig, width=FULL_W, height=400, legend_pos='none',
           yaxis_range=[0, 1.05])
fig.update_xaxes(title_text='', tickangle=-30)
fig.update_yaxes(title_text='Fraction stable (>=2/3 votes)')
save_fig(fig, 'fraction_stable_7m.png')
fig.show()


-- Stability Comparison: original (argmax) vs fixed (u=1.0) --
    Model Stable(orig) Frac(orig) Stable(fix) Frac(fix)
 Historic        21/30        70%       17/27       63%
Heuristic        25/30        83%       16/27       59%
      CST         6/30        20%        7/27       26%
     CGAN        23/30        77%       16/27       59%
    LobS5         6/30        20%        9/27       33%
  S5-120M        10/30        33%        8/27       30%
    S5-4K         7/30        23%       10/27       37%

Note: Original stability code is correct (verified by re-run).
Using original stability_for_folder for the paper.
  Saved: fraction_stable_7m.png


In [18]:
# -- Permanent / Temporary Decomposition --
# (computed here for the no-arb scorecard)

decomp_results = OrderedDict()

for label, r_data in R.items():
    rdf = r_data['relax_df'].copy()
    if rdf.empty or len(rdf) < 3:
        continue

    rdf['I_perm'] = rdf['I_final']
    rdf['I_temp'] = rdf['I_peak'] - rdf['I_final']

    pc = r_data['pc']

    folder_vday = {}
    for f in rdf['folder'].unique():
        pc_f = pc[pc['folder'] == f]
        if not pc_f.empty:
            i_val, c_val, mb_val, V_val = parse_folder_params_v2(f)
            Q_total = i_val * V_val
            last_ins = pc_f[pc_f['insertion_idx'] == pc_f['insertion_idx'].max()]
            if not last_ins.empty:
                med_x = last_ins['x'].median()
                folder_vday[f] = Q_total / np.exp(med_x)

    folder_sigma = {}
    for f in rdf['folder'].unique():
        pc_f = pc[pc['folder'] == f]
        if not pc_f.empty:
            folder_sigma[f] = np.exp(pc_f['alpha'].median())

    valid = []
    for _, row in rdf.iterrows():
        f = row['folder']
        if f not in folder_vday or f not in folder_sigma:
            continue
        V_day = folder_vday[f]
        sigma = folder_sigma[f]
        QV = row['Q'] / V_day
        if QV <= 0 or sigma <= 0:
            continue
        valid.append({
            'folder': f, 'Q': row['Q'], 'V': row['V'],
            'QV': QV, 'sigma': sigma,
            'I_perm': row['I_perm'], 'I_temp': row['I_temp'],
            'I_peak': row['I_peak'],
            'ln_QV': np.log(QV),
            'ln_Iperm_s': np.log(abs(row['I_perm']) / sigma + 1e-30),
            'ln_Itemp_s': np.log(abs(row['I_temp']) / sigma + 1e-30),
        })
    vdf = pd.DataFrame(valid)
    if len(vdf) < 3:
        continue

    ok_p = np.isfinite(vdf['ln_QV']) & np.isfinite(vdf['ln_Iperm_s']) & (vdf['I_perm'] > 0)
    if ok_p.sum() >= 3:
        sl_p = linregress(vdf.loc[ok_p, 'ln_QV'], vdf.loc[ok_p, 'ln_Iperm_s'])
        beta_perm = sl_p.slope
        r2_perm = sl_p.rvalue**2
    else:
        beta_perm, r2_perm = np.nan, np.nan

    ok_t = np.isfinite(vdf['ln_QV']) & np.isfinite(vdf['ln_Itemp_s']) & (vdf['I_temp'] > 0)
    if ok_t.sum() >= 3:
        sl_t = linregress(vdf.loc[ok_t, 'ln_QV'], vdf.loc[ok_t, 'ln_Itemp_s'])
        beta_temp = sl_t.slope
        r2_temp = sl_t.rvalue**2
    else:
        beta_temp, r2_temp = np.nan, np.nan

    decomp_results[label] = {
        'beta_perm': beta_perm, 'r2_perm': r2_perm,
        'beta_temp': beta_temp, 'r2_temp': r2_temp,
        'vdf': vdf,
    }

print("\n-- Permanent/Temporary Decomposition --")
print("  Theory: beta_perm ~ 1.0 (Huberman-Stanzl), beta_temp ~ 0.5")
for label, dr in decomp_results.items():
    print(f"  {label:15s}  beta_perm={dr['beta_perm']:.3f} (R2={dr['r2_perm']:.3f})  "
          f"beta_temp={dr['beta_temp']:.3f} (R2={dr['r2_temp']:.3f})")


-- Permanent/Temporary Decomposition --
  Theory: beta_perm ~ 1.0 (Huberman-Stanzl), beta_temp ~ 0.5
  Historic         beta_perm=0.412 (R2=0.081)  beta_temp=0.921 (R2=0.199)
  Heuristic        beta_perm=0.993 (R2=0.277)  beta_temp=nan (R2=nan)
  CST              beta_perm=0.967 (R2=0.514)  beta_temp=1.418 (R2=0.333)
  CGAN             beta_perm=0.974 (R2=0.345)  beta_temp=1.321 (R2=0.174)
  LobS5            beta_perm=0.982 (R2=0.249)  beta_temp=0.765 (R2=0.260)
  S5-120M          beta_perm=0.562 (R2=0.172)  beta_temp=0.453 (R2=0.058)
  S5-4K            beta_perm=0.587 (R2=0.159)  beta_temp=0.228 (R2=0.010)


In [19]:
# -- Decay Function Fitting --

decay_results = OrderedDict()

for label, r in R.items():
    fits = []
    for f, cv in r['curves'].items():
        dr = fit_decay(cv['u_grid'], cv['combined_mean'])
        if dr is not None and 'gamma' in dr:
            fits.append({
                'folder': f, 'gamma': dr['gamma'],
                'r2_pl': dr.get('r2_pl', np.nan),
                'r2_exp': dr.get('r2_exp', np.nan),
                'aic_pl': dr.get('aic_pl', np.nan),
                'aic_exp': dr.get('aic_exp', np.nan),
                'tau': dr.get('tau', np.nan),
            })
    decay_df = pd.DataFrame(fits) if fits else pd.DataFrame()
    decay_results[label] = decay_df

print("\n-- Decay Fitting Summary --")
for label, ddf in decay_results.items():
    if ddf.empty:
        continue
    g = ddf['gamma']
    print(f"  {label:15s}  gamma={g.mean():.3f}+/-{g.std():.3f} (med={g.median():.3f})  "
          f"AIC_PL<Exp: {(ddf['aic_pl'] < ddf['aic_exp']).sum()}/{len(ddf)}")


-- Decay Fitting Summary --
  Historic         gamma=0.824+/-0.195 (med=0.839)  AIC_PL<Exp: 0/27
  Heuristic        gamma=-1.218+/-4.136 (med=-0.509)  AIC_PL<Exp: 0/26
  CST              gamma=-3.752+/-20.630 (med=0.451)  AIC_PL<Exp: 0/25
  CGAN             gamma=1.534+/-2.664 (med=0.784)  AIC_PL<Exp: 0/26
  LobS5            gamma=0.995+/-3.007 (med=0.456)  AIC_PL<Exp: 0/27
  S5-120M          gamma=-0.464+/-1.913 (med=0.221)  AIC_PL<Exp: 0/24
  S5-4K            gamma=-0.288+/-1.902 (med=0.344)  AIC_PL<Exp: 0/27


In [20]:
# -- No-Arbitrage Consistency Scorecard (5 tests, 7 models) --

arb_rows = []

for label, r_data in R.items():
    delta = r_data['beta']['beta']
    relax_med = r_data['relax_df']['ratio'].median() if not r_data['relax_df'].empty else np.nan

    ddf = decay_results.get(label, pd.DataFrame())
    gamma_med = ddf['gamma'].median() if not ddf.empty else np.nan

    dr = decomp_results.get(label, {})
    beta_perm = dr.get('beta_perm', np.nan)

    # Test A: Concavity (delta < 1)
    test_A = delta < 1.0 if np.isfinite(delta) else False
    # Test B: Permanent impact linearity
    test_B = (0.7 <= beta_perm <= 1.3) if np.isfinite(beta_perm) else False
    # Test C: Decay kernel consistency
    test_C = (0.3 <= gamma_med <= 1.0) if np.isfinite(gamma_med) else False
    # Test D: Relaxation ratio bounds
    test_D = (0.5 <= relax_med <= 1.0) if np.isfinite(relax_med) else False
    # Test E: Gatheral condition
    if np.isfinite(delta) and np.isfinite(gamma_med) and gamma_med > 0:
        gatheral_bound = 1.0 / (1.0 + 2.0 * gamma_med)
        test_E = delta <= gatheral_bound
    else:
        gatheral_bound = np.nan
        test_E = False

    n_pass = sum([test_A, test_B, test_C, test_D, test_E])

    arb_rows.append({
        'Model': label,
        'delta': f'{delta:.3f}',
        'beta_perm': f'{beta_perm:.3f}' if np.isfinite(beta_perm) else '---',
        'gamma': f'{gamma_med:.3f}' if np.isfinite(gamma_med) else '---',
        'relax_r': f'{relax_med:.3f}' if np.isfinite(relax_med) else '---',
        'A:Concav': 'PASS' if test_A else 'FAIL',
        'B:Perm~1': 'PASS' if test_B else 'FAIL',
        'C:Decay': 'PASS' if test_C else 'FAIL',
        'D:Relax': 'PASS' if test_D else 'FAIL',
        'E:Gather': 'PASS' if test_E else 'FAIL',
        'Score': f'{n_pass}/5',
    })

arb_table = pd.DataFrame(arb_rows)
print('\n-- No-Arbitrage Consistency Check (5 tests) --')
print(arb_table.to_string(index=False))

# LaTeX
print('\n-- LaTeX --')
print('\\begin{tabular}{lccccccccc}')
print('\\toprule')
print('Model & $\\delta$ & $\\beta_p$ & $\\gamma$ & $r$ & A & B & C & D & E \\\\')
print('\\midrule')
for _, row in arb_table.iterrows():
    vals = ' & '.join([row['Model'], row['delta'], row['beta_perm'], row['gamma'],
                       row['relax_r'], row['A:Concav'], row['B:Perm~1'],
                       row['C:Decay'], row['D:Relax'], row['E:Gather']])
    print(f'{vals} \\\\')
print('\\bottomrule')
print('\\end{tabular}')


-- No-Arbitrage Consistency Check (5 tests) --
    Model delta beta_perm  gamma relax_r A:Concav B:Perm~1 C:Decay D:Relax E:Gather Score
 Historic 0.525     0.412  0.839   0.073     PASS     FAIL    PASS    FAIL     FAIL   2/5
Heuristic 0.524     0.993 -0.509   1.479     PASS     PASS    FAIL    FAIL     FAIL   2/5
      CST 0.575     0.967  0.451   0.818     PASS     PASS    PASS    PASS     FAIL   4/5
     CGAN 0.574     0.974  0.784   0.983     PASS     PASS    PASS    PASS     FAIL   4/5
    LobS5 0.566     0.982  0.456   0.641     PASS     PASS    PASS    PASS     FAIL   4/5
  S5-120M 0.534     0.562  0.221   0.895     PASS     FAIL    FAIL    PASS     PASS   3/5
    S5-4K 0.546     0.587  0.344   0.877     PASS     FAIL    PASS    PASS     PASS   4/5

-- LaTeX --
\begin{tabular}{lccccccccc}
\toprule
Model & $\delta$ & $\beta_p$ & $\gamma$ & $r$ & A & B & C & D & E \\
\midrule
Historic & 0.525 & 0.412 & 0.839 & 0.073 & PASS & FAIL & PASS & FAIL & FAIL \\
Heuristic & 0.524 & 0.993

In [21]:
# -- Figure: S5 Variant Master Curves Overlay --

s5_labels = [l for l in R if l in MODEL_META]
fig = go.Figure()

u_max_s5 = []
for label in s5_labels:
    for f, c in R[label]['curves'].items():
        u_max_s5.append(c['u_grid'][-1])
U_PLOT_S5 = min(3.0, np.percentile(u_max_s5, 10)) if u_max_s5 else 3.0
u_common = np.linspace(0, U_PLOT_S5, 300)

for label in s5_labels:
    r = R[label]
    interps = []
    for f, c in r['curves'].items():
        if c['u_grid'][-1] >= U_PLOT_S5:
            interps.append(np.interp(u_common, c['u_grid'], c['combined_mean']))
    if not interps:
        continue
    avg = np.mean(interps, axis=0)
    std = np.std(interps, axis=0)
    color = SCENARIOS[label]['color']
    rc, gc_, bc = int(color[1:3],16), int(color[3:5],16), int(color[5:7],16)
    fill_rgba = f'rgba({rc},{gc_},{bc},0.12)'

    fig.add_trace(go.Scatter(
        x=np.concatenate([u_common, u_common[::-1]]),
        y=np.concatenate([avg + std, (avg - std)[::-1]]),
        fill='toself', fillcolor=fill_rgba,
        line=dict(width=0), showlegend=False))

    mm = MODEL_META[label]
    lbl = f"{label} ({mm['params']/1e6:.0f}M, {mm['context']}ctx)"
    fig.add_trace(go.Scatter(
        x=u_common, y=avg, mode='lines',
        line=dict(color=color, width=2.5), name=lbl))

fig.add_vline(x=1.0, line_dash='dot', line_color='rgba(0,0,0,0.35)', line_width=1)

pub_layout(fig, width=FULL_W, height=420, legend_pos='tr')
fig.update_xaxes(title_text='u = n / L')
fig.update_yaxes(title_text='I<sub>norm</sub>(u)')
save_fig(fig, 's5_variants_overlay_7m.png')
fig.show()

  Saved: s5_variants_overlay_7m.png


---
## Part B: Appendix

### A: Per-Day Beta
### B: Decay Function Fitting
### C: Permanent/Temporary Decomposition
### D: Bootstrap Distributions (already shown above)
### E: Stability Details

In [22]:
# -- Appendix A: Per-Day Beta --

perday_results = OrderedDict()

for label, r in R.items():
    pc = r['pc']
    if pc.empty:
        continue
    sdm = SAMPLE_DAY_MAP[['sample_id', 'day']].drop_duplicates()
    pc_day = pc.merge(sdm, on='sample_id', how='left')
    day_betas = []
    for day_val, grp in pc_day.groupby('day'):
        bstat = compute_global_beta(grp)
        if np.isfinite(bstat['beta']):
            day_betas.append({'day': day_val, 'beta': bstat['beta'],
                              'r2': bstat['r2'], 'n': bstat['n']})
    day_df = pd.DataFrame(day_betas)
    perday_results[label] = day_df

rows = []
for label, ddf in perday_results.items():
    if ddf.empty:
        continue
    b = ddf['beta']
    rows.append({'Model': label, 'Mean beta': f'{b.mean():.3f}',
                 'Std': f'{b.std():.3f}', 'Min': f'{b.min():.3f}',
                 'Max': f'{b.max():.3f}', 'N_days': len(ddf)})
print('\n-- Per-Day Beta --')
print(pd.DataFrame(rows).to_string(index=False))

# Figure
fig = go.Figure()
for label, ddf in perday_results.items():
    if ddf.empty:
        continue
    fig.add_trace(go.Box(
        y=ddf['beta'], name=label,
        marker_color=SCENARIOS[label]['color'],
        line_color=SCENARIOS[label]['color'],
        boxpoints='all', jitter=0.3, pointpos=-1.5,
        marker=dict(size=5, opacity=0.7),
        line_width=1.5))
fig.add_hline(y=0.5, line_dash='dash', line_color='black', line_width=1.5,
              annotation_text='\u03b2 = 0.5', annotation_font_size=11,
              annotation_position='bottom right')
pub_layout(fig, width=FULL_W, height=400, legend_pos='none')
fig.update_xaxes(title_text='', tickangle=-30)
fig.update_yaxes(title_text='\u03b2 (per day)')
save_fig(fig, 'perday_beta_7m.png')
fig.show()


-- Per-Day Beta --
    Model Mean beta   Std   Min   Max  N_days
 Historic     0.525 0.017 0.507 0.552       8
Heuristic     0.524 0.016 0.500 0.551       8
      CST     0.576 0.037 0.505 0.621       8
     CGAN     0.571 0.019 0.546 0.600       8
    LobS5     0.568 0.019 0.543 0.592       8
  S5-120M     0.530 0.057 0.440 0.615       8
    S5-4K     0.541 0.040 0.469 0.609       8
  Saved: perday_beta_7m.png


In [23]:
# -- Appendix B: Decay Function Fitting --

rows = []
for label, ddf in decay_results.items():
    if ddf.empty:
        continue
    g = ddf['gamma']
    rows.append({
        'Model': label,
        'gamma_mean': f'{g.mean():.3f}',
        'gamma_std': f'{g.std():.3f}',
        'gamma_med': f'{g.median():.3f}',
        'R2_PL': f"{ddf['r2_pl'].mean():.3f}",
        'R2_Exp': f"{ddf['r2_exp'].mean():.3f}",
        'AIC_PL<Exp': f"{(ddf['aic_pl'] < ddf['aic_exp']).sum()}/{len(ddf)}",
    })
print('\n-- Decay Fitting: Power-Law gamma --')
print('  Expected: gamma in [0.5, 0.8] (Brokmann 2015)')
print(pd.DataFrame(rows).to_string(index=False))

# Figure: decay exponent distribution
fig = go.Figure()
for label, ddf in decay_results.items():
    if ddf.empty:
        continue
    fig.add_trace(go.Box(
        y=ddf['gamma'], name=label,
        marker_color=SCENARIOS[label]['color'],
        line_color=SCENARIOS[label]['color'],
        boxpoints='all', jitter=0.3, pointpos=-1.5,
        marker=dict(size=4, opacity=0.5),
        line_width=1.5))
fig.add_hline(y=0.5, line_dash='dash', line_color='black', line_width=1,
              annotation_text='\u03b3=0.5', annotation_font_size=10,
              annotation_position='bottom right')
fig.add_hline(y=0.8, line_dash='dash', line_color='grey', line_width=1,
              annotation_text='\u03b3=0.8', annotation_font_size=10,
              annotation_position='top right')
pub_layout(fig, width=FULL_W, height=400, legend_pos='none')
fig.update_xaxes(title_text='', tickangle=-30)
fig.update_yaxes(title_text='\u03b3 (decay exponent)')
save_fig(fig, 'decay_exponent_7m.png')
fig.show()

# Figure: example decay fits
n_scn = len(R)
n_cols = min(n_scn, 4)
n_rows_d = math.ceil(n_scn / n_cols)
fig = make_subplots(rows=n_rows_d, cols=n_cols,
    subplot_titles=[f'<b>{l}</b>' for l in R.keys()],
    horizontal_spacing=0.10, vertical_spacing=0.12)

for idx, (label, r) in enumerate(R.items()):
    row, col = idx // n_cols + 1, idx % n_cols + 1
    ddf = decay_results[label]
    if ddf.empty:
        continue
    med_idx = (ddf['gamma'] - ddf['gamma'].median()).abs().idxmin()
    folder = ddf.loc[med_idx, 'folder']
    cv = r['curves'][folder]
    dr = fit_decay(cv['u_grid'], cv['combined_mean'])
    if dr is None:
        continue
    u_post = dr['u_post']
    fig.add_trace(go.Scatter(x=u_post, y=dr['I_post'], mode='lines',
        line=dict(color=SCENARIOS[label]['color'], width=2),
        name='Data', showlegend=(idx==0)), row=row, col=col)
    if 'I_fit_pl' in dr:
        fig.add_trace(go.Scatter(x=u_post, y=dr['I_fit_pl'], mode='lines',
            line=dict(color='red', width=1.5, dash='dash'),
            name='Power-law', showlegend=(idx==0)), row=row, col=col)
    if 'I_fit_exp' in dr:
        fig.add_trace(go.Scatter(x=u_post, y=dr['I_fit_exp'], mode='lines',
            line=dict(color='blue', width=1.5, dash='dot'),
            name='Exponential', showlegend=(idx==0)), row=row, col=col)

fig.update_layout(
    width=FULL_W, height=int(FULL_W * 0.35 * n_rows_d),
    template='plotly_white',
    font=dict(family='Times New Roman, DejaVu Serif, serif', size=12, color='black'),
    margin=dict(l=55, r=15, t=35, b=50),
    legend=dict(x=0.98, y=0.98, xanchor='right', yanchor='top',
                bgcolor='rgba(255,255,255,0.85)', bordercolor='black', borderwidth=1, font_size=10),
)
fig.update_xaxes(**_AX, title_text='u', title_font_size=11)
fig.update_yaxes(**_AX, title_text='I<sub>norm</sub>(u)', title_font_size=11)
save_fig(fig, 'decay_fits_example_7m.png')
fig.show()


-- Decay Fitting: Power-Law gamma --
  Expected: gamma in [0.5, 0.8] (Brokmann 2015)
    Model gamma_mean gamma_std gamma_med    R2_PL R2_Exp AIC_PL<Exp
 Historic      0.824     0.195     0.839  -14.756  0.858       0/27
Heuristic     -1.218     4.136    -0.509   -0.050  0.668       0/26
      CST     -3.752    20.630     0.451   -1.106  0.490       0/25
     CGAN      1.534     2.664     0.784 -231.877  0.448       0/26
    LobS5      0.995     3.007     0.456   -4.163  0.526       0/27
  S5-120M     -0.464     1.913     0.221   -2.184  0.472       0/24
    S5-4K     -0.288     1.902     0.344   -2.344  0.503       0/27
  Saved: decay_exponent_7m.png


  Saved: decay_fits_example_7m.png


In [24]:
# -- Appendix C: Permanent/Temporary Decomposition --

rows = []
for label, dr in decomp_results.items():
    rows.append({
        'Model': label,
        'beta_perm': f"{dr['beta_perm']:.3f}",
        'R2_perm': f"{dr['r2_perm']:.3f}",
        'beta_temp': f"{dr['beta_temp']:.3f}",
        'R2_temp': f"{dr['r2_temp']:.3f}",
    })
print('\n-- Permanent/Temporary Decomposition --')
print('  Theory: beta_perm ~ 1.0 (Huberman-Stanzl), beta_temp ~ 0.5')
print(pd.DataFrame(rows).to_string(index=False))

# Figure: perm/temp log-log scatter
fig = make_subplots(rows=1, cols=2,
    subplot_titles=['<b>Permanent (I<sub>final</sub>)</b>',
                    '<b>Temporary (I<sub>peak</sub> - I<sub>final</sub>)</b>'],
    horizontal_spacing=0.15)

for label, dr in decomp_results.items():
    vdf = dr['vdf']
    color = SCENARIOS[label]['color']
    ok_p = (vdf['I_perm'] > 0)
    ok_t = (vdf['I_temp'] > 0)
    if ok_p.sum() >= 2:
        fig.add_trace(go.Scatter(
            x=vdf.loc[ok_p, 'ln_QV'], y=vdf.loc[ok_p, 'ln_Iperm_s'],
            mode='markers', marker=dict(size=5, color=color, opacity=0.6),
            name=f"{label}", showlegend=True), row=1, col=1)
        x_r = np.array([vdf['ln_QV'].min(), vdf['ln_QV'].max()])
        sl = linregress(vdf.loc[ok_p, 'ln_QV'], vdf.loc[ok_p, 'ln_Iperm_s'])
        fig.add_trace(go.Scatter(
            x=x_r, y=sl.slope * x_r + sl.intercept, mode='lines',
            line=dict(color=color, width=1.5, dash='dash'),
            showlegend=False), row=1, col=1)
    if ok_t.sum() >= 2:
        fig.add_trace(go.Scatter(
            x=vdf.loc[ok_t, 'ln_QV'], y=vdf.loc[ok_t, 'ln_Itemp_s'],
            mode='markers', marker=dict(size=5, color=color, opacity=0.6),
            showlegend=False), row=1, col=2)
        sl = linregress(vdf.loc[ok_t, 'ln_QV'], vdf.loc[ok_t, 'ln_Itemp_s'])
        fig.add_trace(go.Scatter(
            x=x_r, y=sl.slope * x_r + sl.intercept, mode='lines',
            line=dict(color=color, width=1.5, dash='dash'),
            showlegend=False), row=1, col=2)

fig.update_layout(
    width=FULL_W, height=420,
    template='plotly_white',
    font=dict(family='Times New Roman, DejaVu Serif, serif', size=12, color='black'),
    margin=dict(l=55, r=15, t=35, b=55),
    legend=dict(x=0.45, y=0.02, xanchor='center', yanchor='bottom',
                bgcolor='rgba(255,255,255,0.85)', bordercolor='black', borderwidth=1,
                font_size=10, orientation='h'),
)
fig.update_xaxes(**_AX, title_text='ln(Q/V)')
fig.update_yaxes(**_AX)
fig.update_yaxes(title_text='ln(I<sub>perm</sub>/\u03c3)', row=1, col=1)
fig.update_yaxes(title_text='ln(I<sub>temp</sub>/\u03c3)', row=1, col=2)
save_fig(fig, 'perm_temp_decomp_7m.png')
fig.show()


-- Permanent/Temporary Decomposition --
  Theory: beta_perm ~ 1.0 (Huberman-Stanzl), beta_temp ~ 0.5
    Model beta_perm R2_perm beta_temp R2_temp
 Historic     0.412   0.081     0.921   0.199
Heuristic     0.993   0.277       nan     nan
      CST     0.967   0.514     1.418   0.333
     CGAN     0.974   0.345     1.321   0.174
    LobS5     0.982   0.249     0.765   0.260
  S5-120M     0.562   0.172     0.453   0.058
    S5-4K     0.587   0.159     0.228   0.010
  Saved: perm_temp_decomp_7m.png


In [25]:
# -- Appendix: Gamma (Volume Scaling Exponent) --
# NOTE: gamma measures per-order microstructural impact, NOT meta-order
# square-root law. Super-linear gamma is expected (larger orders eat
# through multiple book levels). Only 3 data points per regression.

fig = go.Figure()
for label, r in R.items():
    gdf = r['gamma_df']
    if gdf.empty:
        continue
    fig.add_trace(go.Box(
        y=gdf['gamma'], name=label,
        marker_color=SCENARIOS[label]['color'],
        line_color=SCENARIOS[label]['color'],
        boxpoints='all', jitter=0.3, pointpos=-1.5,
        marker=dict(size=4, opacity=0.5),
        line_width=1.5))

fig.add_hline(y=0.5, line_dash='dash', line_color='black', line_width=1.5,
              annotation_text='\u03b3 = 0.5', annotation_font_size=11,
              annotation_position='bottom right')

pub_layout(fig, width=FULL_W, height=400, legend_pos='none')
fig.update_xaxes(title_text='', tickangle=-30)
fig.update_yaxes(title_text='\u03b3 (volume scaling exponent)')
save_fig(fig, 'gamma_distribution_7m.png')
fig.show()

for label, r in R.items():
    gdf = r['gamma_df']
    if not gdf.empty:
        print(f"{label:15s}  gamma = {gdf['gamma'].mean():.3f} +/- {gdf['gamma'].std():.3f}  (n={len(gdf)})")

  Saved: gamma_distribution_7m.png


Historic         gamma = 0.942 +/- 0.117  (n=10)
Heuristic        gamma = 0.925 +/- 0.124  (n=10)
CST              gamma = 0.708 +/- 0.252  (n=10)
CGAN             gamma = 0.718 +/- 0.081  (n=10)
LobS5            gamma = 0.780 +/- 0.222  (n=10)
S5-120M          gamma = 0.430 +/- 0.326  (n=10)
S5-4K            gamma = 0.571 +/- 0.168  (n=10)


---
## Part C: Grand Summary

In [26]:
# -- Grand Summary (7 models) --

print("=" * 115)
hdr = (f"{'Model':15s}  {'Params':>7s}  {'beta':>6s}  {'R2':>6s}  {'Relax':>6s}  "
       f"{'Rlx@u3':>7s}  {'Stable':>7s}  {'gamma':>6s}  {'decay':>6s}  "
       f"{'b_perm':>6s}  {'b_temp':>6s}  {'Arb':>5s}")
print(hdr)
print("-" * 115)
for label in R:
    beta = R[label]['beta']['beta']
    r2 = R[label]['beta']['r2']
    relax_med = R[label]['relax_df']['ratio'].median() if not R[label]['relax_df'].empty else np.nan
    relax_u3 = R[label]['relax_df_fixed']['ratio_u3'].median() if not R[label]['relax_df_fixed'].empty else np.nan
    stable = R[label]['stab_df']['stabilized'].mean() if not R[label]['stab_df'].empty else 0
    gamma_m = R[label]['gamma_df']['gamma'].mean() if not R[label]['gamma_df'].empty else np.nan

    ddf = decay_results.get(label, pd.DataFrame())
    gamma_dec = ddf['gamma'].median() if not ddf.empty else np.nan

    dr = decomp_results.get(label, {})
    bp = dr.get('beta_perm', np.nan)
    bt = dr.get('beta_temp', np.nan)

    arb = [r for r in arb_rows if r['Model'] == label]
    score = arb[0]['Score'] if arb else '---'

    mm = MODEL_META.get(label, {})
    params_str = f"{mm['params']/1e6:.0f}M" if 'params' in mm else '---'

    print(f"{label:15s}  {params_str:>7s}  {beta:6.3f}  {r2:6.3f}  {relax_med:6.3f}  "
          f"{relax_u3:7.3f}  {stable:6.0%}  {gamma_m:6.3f}  {gamma_dec:6.3f}  "
          f"{bp:6.3f}  {bt:6.3f}  {score:>5s}")
print("=" * 115)
print(f"{'Theory':15s}  {'':>7s}  {'0.500':>6s}  {'':>6s}  {'0.667':>6s}  "
      f"{'0.667':>7s}  {'':>7s}  {'':>6s}  {'0.5-8':>6s}  "
      f"{'1.000':>6s}  {'0.500':>6s}  {'5/5':>5s}")

Model             Params    beta      R2   Relax   Rlx@u3   Stable   gamma   decay  b_perm  b_temp    Arb
-------------------------------------------------------------------------------------------------------------------
Historic             ---   0.525   0.956   0.073    0.383     70%   0.942   0.839   0.412   0.921    2/5
Heuristic            ---   0.524   0.957   1.479    1.321     83%   0.925  -0.509   0.993     nan    2/5
CST                  ---   0.575   0.957   0.818    1.019     20%   0.708   0.451   0.967   1.418    4/5
CGAN                 ---   0.574   0.990   0.983    1.035     77%   0.718   0.784   0.974   1.321    4/5
LobS5                45M   0.566   0.954   0.641    0.728     20%   0.780   0.456   0.982   0.765    4/5
S5-120M             120M   0.534   0.925   0.895    0.899     33%   0.430   0.221   0.562   0.453    3/5
S5-4K                55M   0.546   0.939   0.877    0.842     23%   0.571   0.344   0.587   0.228    4/5
Theory                     0.500           

In [27]:
# -- Figure: No-Arbitrage Scatter (beta_perm vs relaxation) --

fig = go.Figure()

# Acceptable zone
fig.add_shape(type='rect', x0=0.7, x1=1.3, y0=0.5, y1=1.0,
              fillcolor='rgba(0,200,0,0.08)', line=dict(color='green', width=1, dash='dot'))

# Theoretical target
fig.add_trace(go.Scatter(
    x=[1.0], y=[2/3], mode='markers',
    marker=dict(symbol='star', size=18, color='gold', line=dict(color='black', width=1.5)),
    name='Theory (1.0, 2/3)', showlegend=True))

for label in R:
    dr = decomp_results.get(label, {})
    bp = dr.get('beta_perm', np.nan)
    rdf = R[label]['relax_df']
    relax_med = rdf['ratio'].median() if not rdf.empty else np.nan
    if not np.isfinite(bp) or not np.isfinite(relax_med):
        continue
    fig.add_trace(go.Scatter(
        x=[bp], y=[relax_med], mode='markers+text',
        marker=dict(size=12, color=SCENARIOS[label]['color'],
                    line=dict(color='black', width=1)),
        text=[label], textposition='top center', textfont=dict(size=10),
        name=label, showlegend=False))

pub_layout(fig, width=SINGLE_W, height=SINGLE_W, legend_pos='bl')
fig.update_xaxes(title_text='\u03b2<sub>perm</sub>')
fig.update_yaxes(title_text='Relaxation ratio r')
save_fig(fig, 'noarb_scatter_7m.png')
fig.show()

  Saved: noarb_scatter_7m.png
